# 21a — Transfer Learning and Pre-trained Models

Corresponds to: Note 21 — Transfer Learning and Pre-trained Models

**Libraries introduced in this notebook:** `torchvision.models` — provides pre-trained CNN architectures (ResNet18) with ImageNet weights; `torchvision.datasets` — provides CIFAR-10 with automatic download and standard transforms.

**What this notebook demonstrates:**
1. Loading CIFAR-10 and preparing it for a pre-trained ResNet18 (resize to 224×224, normalise with ImageNet stats)
2. Loading ResNet18 with pre-trained ImageNet weights
3. Feature extraction — freeze the backbone, replace the FC head, train only the head
4. Fine-tuning — unfreeze the last residual block, continue training at a lower learning rate
5. Comparing trainable parameter counts across both strategies

## Part 1: Setup and Data Loading

CIFAR-10 has 32×32 RGB images. ResNet18 was trained on 224×224 ImageNet images. We resize to 224×224 and normalise using the ImageNet mean and standard deviation — the same statistics the pre-trained weights expect.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_full = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_full  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

torch.manual_seed(42)
train_idx = torch.randperm(len(train_full))[:5000]
val_idx   = torch.randperm(len(test_full))[:1000]

train_set = Subset(train_full, train_idx)
val_set   = Subset(test_full,  val_idx)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=0)

print(f'Training samples:   {len(train_set)}')
print(f'Validation samples: {len(val_set)}')
print(f'Classes: {train_full.classes}')

## Part 2: Loading Pre-trained ResNet18

ResNet18 is loaded with weights trained on ImageNet. We inspect its architecture and parameter count before any modification.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

total_params = sum(p.numel() for p in model.parameters())
print(f'ResNet18 total parameters: {total_params:,}')
print()
print('Final FC layer (original):')
print(f'  {model.fc}')
print(f'  Input features: {model.fc.in_features}')
print(f'  Output classes: {model.fc.out_features}')

## Part 3: Feature Extraction

Freeze all backbone parameters, replace the FC layer with a new head for 10 CIFAR-10 classes. Only the new head is trained.

In [ ]:
model_fe = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in model_fe.parameters():
    param.requires_grad = False

model_fe.fc = nn.Linear(512, 10)
model_fe = model_fe.to(device)

trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model_fe.parameters() if not p.requires_grad)
print(f'Feature extraction setup:')
print(f'  Trainable parameters: {trainable:,}')
print(f'  Frozen parameters:    {frozen:,}')
print(f'  Trainable fraction:   {trainable / (trainable + frozen) * 100:.2f}%')

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

criterion    = nn.CrossEntropyLoss()
optimizer_fe = optim.Adam(filter(lambda p: p.requires_grad, model_fe.parameters()), lr=1e-3)

print('Feature Extraction Training (5 epochs):')
print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Val Loss":<14} {"Val Acc"}')
print('-' * 60)

for epoch in range(1, 6):
    tr_loss, tr_acc = train_epoch(model_fe, train_loader, criterion, optimizer_fe)
    va_loss, va_acc = eval_epoch(model_fe, val_loader, criterion)
    print(f'{epoch:<8} {tr_loss:<14.4f} {tr_acc:<14.4f} {va_loss:<14.4f} {va_acc:.4f}')

## Part 4: Fine-tuning

Unfreeze Layer 4 (the last residual block) and continue training. The backbone uses lr=1e-5 to preserve pre-trained features; the head uses lr=1e-3.

In [ ]:
for param in model_fe.layer4.parameters():
    param.requires_grad = True

trainable_ft = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
frozen_ft    = sum(p.numel() for p in model_fe.parameters() if not p.requires_grad)
print(f'Fine-tuning setup (layer4 + head unfrozen):')
print(f'  Trainable parameters: {trainable_ft:,}')
print(f'  Frozen parameters:    {frozen_ft:,}')
print(f'  Trainable fraction:   {trainable_ft / (trainable_ft + frozen_ft) * 100:.2f}%')

optimizer_ft = optim.Adam([
    {'params': model_fe.layer4.parameters(), 'lr': 1e-5},
    {'params': model_fe.fc.parameters(),     'lr': 1e-3},
])

print()
print('Fine-tuning Training (5 more epochs):')
print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Val Loss":<14} {"Val Acc"}')
print('-' * 60)

for epoch in range(1, 6):
    tr_loss, tr_acc = train_epoch(model_fe, train_loader, criterion, optimizer_ft)
    va_loss, va_acc = eval_epoch(model_fe, val_loader, criterion)
    print(f'{epoch:<8} {tr_loss:<14.4f} {tr_acc:<14.4f} {va_loss:<14.4f} {va_acc:.4f}')

## Part 5: Parameter Count Comparison

In [ ]:
model_scratch = models.resnet18(weights=None)
model_scratch.fc = nn.Linear(512, 10)
total = sum(p.numel() for p in model_scratch.parameters())
fe_trainable = 512 * 10 + 10  # just the FC layer

print('Parameter count comparison:')
print(f'{"Strategy":<30} {"Trainable":>12} {"% of total":>12}')
print('-' * 56)
print(f'{"Train from scratch":<30} {total:>12,} {100.0:>11.1f}%')
print(f'{"Feature extraction":<30} {fe_trainable:>12,} {fe_trainable/total*100:>11.2f}%')
print(f'{"Fine-tune (layer4+head)":<30} {trainable_ft:>12,} {trainable_ft/total*100:>11.2f}%')

## Summary

What this notebook demonstrated:

1. **Data preparation** — CIFAR-10 resized to 224×224 and normalised with ImageNet statistics so the pre-trained ResNet18 receives inputs in the expected range.
2. **Pre-trained weights** — ResNet18 loaded with `IMAGENET1K_V1` weights. The original FC layer (512→1000) is replaced with a new head (512→10) for CIFAR-10.
3. **Feature extraction** — all backbone parameters frozen (`requires_grad=False`). Only the new FC head (5,130 parameters, 0.04% of total) is trained.
4. **Fine-tuning** — Layer 4 unfrozen and trained at lr=1e-5 while the head trains at lr=1e-3. Two parameter groups passed to the optimiser with different learning rates.
5. **Parameter counts** — feature extraction trains 0.04% of parameters; fine-tuning trains ~72% (layer4 + head). Training from scratch trains 100% from random initialisation.